# ME 280 HW 1

I am starting to like these workbooks. Though, I would still like it to be 100% markdown with runnable block-code cells. Anyway, before I do anything, I wanted to create an enum for all cards. According to [the official docs](https://docs.python.org/3/library/enum.html), there's no enum keyword in this language but there is an abstract class.


In [3774]:
from enum import Enum


# enums are just class inherence here, nothing special here

# I personally like CamelCase for the entries but it's acceptable
# in my eyes to just follow the docs and do SCREAMING_SNAKE_CASE

# Oh also, the numbers here are arbitrary, they're not the actual values


class Card(Enum):

    ACE = 1
    TWO = 2
    THREE = 3
    FOUR = 4
    FIVE = 5
    SIX = 6
    SEVEN = 7
    EIGHT = 8
    NINE = 9
    TEN = 10
    JACK = 11
    QUEEN = 12
    KING = 13

    # declaring a worth method here since the indices are arbitrary
    def worth(self, treat_ace_as_1=True) -> int:

        # first and foremost, we handle the ace
        if self == Card.ACE:
            # the ace of course can either be 1 or 11 depending of the user's choice
            return 1 if treat_ace_as_1 else 11

        # then we handle the face cards which are all 10
        elif self in [Card.JACK, Card.QUEEN, Card.KING]:
            return 10

        # the rest actually have the correct values
        else:
            return self.value

    def __str__(self):
        case = {
            Card.ACE: "🂱",
            Card.TWO: "🂲",
            Card.THREE: "🂳",
            Card.FOUR: "🂴",
            Card.FIVE: "🂵",
            Card.SIX: "🂶",
            Card.SEVEN: "🂷",
            Card.EIGHT: "🂸",
            Card.NINE: "🂹",
            Card.TEN: "🂺",
            Card.JACK: "🂻",
            Card.QUEEN: "🂽",
            Card.KING: "🂾",
        }

        return case[self]

It would be a great idea to test all these methods and variants out.


In [3775]:
# testing all cases
print(Card.ACE, Card.ACE.worth())
print(Card.ACE, Card.ACE.worth(False))
print(Card.TWO, Card.TWO.worth())
print(Card.THREE, Card.THREE.worth())
print(Card.FOUR, Card.FOUR.worth())
print(Card.FIVE, Card.FIVE.worth())
print(Card.SIX, Card.SIX.worth())
print(Card.SEVEN, Card.SEVEN.worth())
print(Card.EIGHT, Card.EIGHT.worth())
print(Card.NINE, Card.NINE.worth())
print(Card.TEN, Card.TEN.worth())
print(Card.JACK, Card.JACK.worth())
print(Card.QUEEN, Card.QUEEN.worth())
print(Card.KING, Card.KING.worth())

🂱 1
🂱 11
🂲 2
🂳 3
🂴 4
🂵 5
🂶 6
🂷 7
🂸 8
🂹 9
🂺 10
🂻 10
🂽 10
🂾 10


In [3776]:
class PlayerBehavior(Enum):
    MORON_PLAYER = 0
    SMART_PLAYER = 1
    DEALER = 2

In [3777]:
import random


class Player:
    def __init__(
        self,
        name: str,
        behavior=PlayerBehavior.MORON_PLAYER,
        initial_cards=0,
        treat_ace_as_1=True,
    ):
        self.name = name
        self.behavior = behavior
        self.standing = False

        self.cards: list[Card] = []
        self.hit(initial_cards)

        self.treat_ace_as_1 = treat_ace_as_1

    def title_string(self):
        return f"{self.name}: {self.worth()}"

    def deck_string(self):
        return " ".join(str(card) for card in self.cards)

    def draw_random():
        return random.choice(list(Card))

    def upcard(self):
        return self.cards[0]

    def hit(self, hits=1):
        for _ in range(hits):
            self.cards.append(Player.draw_random())

    def stand(self):
        self.standing = True

    def worth(self):
        return sum(card.worth(self.treat_ace_as_1) for card in self.cards)

    def is_bust(self):
        return self.worth() > 21

    def decide(self, opponent: "Player"):
        if self.behavior == PlayerBehavior.MORON_PLAYER:
            if random.random() < 0.5:
                self.hit()
            else:
                self.stand()

        elif self.behavior == PlayerBehavior.SMART_PLAYER:
            values = [card.worth(self.treat_ace_as_1) for card in Card]
            average = sum(values) / len(values)

            if self.worth() < 21 - average:
                self.hit()
            else:
                self.stand()

        elif self.behavior == PlayerBehavior.DEALER:
            if self.worth() < 17:
                self.hit()
            else:
                self.stand()

    def play(self, opponent: "Player"):
        while not self.standing and not self.is_bust():
            self.decide(opponent)

In [3778]:
class GameResult(Enum):
    PLAYER_WINNER = 0
    DEALER_WINNER = 1
    DRAW = 2

In [3779]:
class BlackJack:
    def __init__(
        self, player_behavior=PlayerBehavior.MORON_PLAYER, treat_ace_as_1=True
    ):
        self.player = Player(
            "Player",
            behavior=player_behavior,
            initial_cards=2,
            treat_ace_as_1=treat_ace_as_1,
        )
        self.dealer = Player(
            "Dealer",
            behavior=PlayerBehavior.DEALER,
            initial_cards=1,
            treat_ace_as_1=treat_ace_as_1,
        )

    def play(self):
        self.player.play(self.dealer)

        if self.player.is_bust():
            return GameResult.DEALER_WINNER

        self.dealer.play(self.player)

        if self.dealer.is_bust() or self.player.worth() > self.dealer.worth():
            return GameResult.PLAYER_WINNER

        if self.player.worth() < self.dealer.worth():
            return GameResult.DEALER_WINNER

        return GameResult.DRAW

    def play_and_print(self):
        result = self.play()

        if result == GameResult.PLAYER_WINNER:
            print(f"🟢 {self.player.name} wins!")
        elif result == GameResult.DEALER_WINNER:
            print(f"🔴 {self.dealer.name} wins")
        else:
            print("🟡 Draw")

        player_title = self.player.title_string()
        player_deck = self.player.deck_string()
        print(
            f"   {player_title}{" " * (24 - len(player_title))}{self.dealer.title_string()}"
        )
        print(
            f"   {player_deck}{" " * (24 - len(player_deck))}{self.dealer.deck_string()}",
            end="\n\n",
        )

In [3780]:
games = 30_000

moron_won = 0
moron_drawn = 0

for _ in range(games):
    result = BlackJack(player_behavior=PlayerBehavior.MORON_PLAYER).play()

    if result == GameResult.PLAYER_WINNER:
        moron_won += 1

    if result == GameResult.DRAW:
        moron_drawn += 1

print((moron_won / (games - moron_drawn)) * 100)

29.21945662305382


In [3781]:
smart_won = 0
smart_drawn = 0

for _ in range(games):
    result = BlackJack(player_behavior=PlayerBehavior.SMART_PLAYER).play()

    if result == GameResult.PLAYER_WINNER:
        smart_won += 1

    if result == GameResult.DRAW:
        smart_drawn += 1

print((smart_won / (games - smart_drawn)) * 100)

45.61747206501233
